In [1]:
from parselog import *

In [2]:
lp = LogParser(
    log="results/org_setup-cap_ptr-l1/em3d/sim_stdout.gz", 
    lineTypesToPrune=[None, NonRVFILine],
    # If a timestamped line matches, then either RVFILine or NonRVFILine will match
    lineTypesToError=[TimestampedLine],
    RootLogLine=TimestampedLine,
)

Loaded 4438313 log lines
	LLCRqCreationLine:            42670 instances
	LLCRqMissLine:                9868 instances
	LLCRqHitLine:                 42670 instances
	RVFILine:                     2324570 instances
	CRqCreationLine:              543616 instances
	CRqMissLine:                  42314 instances
	CRqHitLine:                   542732 instances
	CRqHitDataLine:               325957 instances
	CRqDependencyLine:            8995 instances
	CapPtrCacheDataArrivalLine:   310103 instances
	CapPtrDataAddTTableEntryLine: 135021 instances
	CapPtrDataLookupPTableLine:   26217 instances
	CapPtrPTUpgradeLine:          33857 instances
	CapPtrCanPrefetchLine:        24702 instances
	CapPtrTLBResponse:            24702 instances
	PRqLine:                      319 instances
Pre-processed 4438313 log lines
Post-processed 4438313 log lines
End-processed 4438313 log lines
Accumulated totals and dists for 4438313 log lines


In [3]:
lp.printTotals()

LLCRqCreationLine totals:
	total: 42670
	discard: 0
	demand: 42670
	demandHit: 32802
	demandMiss: 9868
	demandOwned: 0
	prefetch: 0
	prefetchHit: 0
	prefetchMiss: 0
	prefetchOwned: 0
	usefulPrefetch: 0
	uselessPrefetch: 0

LLCRqMissLine totals:
	total: 9868
	discard: 0
	warning(No hit for LL cRq): 9868

LLCRqHitLine totals:
	total: 42670
	discard: 0
	warning(no LL cRq eviction for miss): 9868

RVFILine totals:
	total: 2324570
	discard: 0

CRqCreationLine totals:
	total: 543616
	discard: 0
	Pr: 24528
	St: 193131
	demand: 519088
	demandHit: 486302
	demandMiss: 24675
	demandMissLL: 6778
	demandOwned: 8111
	demandQueued: 0
	prefetch: 24528
	prefetchHit: 6717
	prefetchMiss: 16927
	prefetchMissLL: 2773
	prefetchOwned: 884
	latePrefetch: 958
	latePrefetchCreation: 3
	latePrefetchIssue: 955
	lateUsefulPrefetch: 243
	prefUnderPref: 359
	usefulPrefetch: 703
	uselessPrefetch: 16224
	uselessPrefetchBecausePerms: 2965
	uselessPrefetchDisruption: 1298
	prefetchDisruption: 1366
	Ld: 325957

CRqMissLi

In [4]:
useless = []
useful = []
for ll in lp.logLines:
    if isinstance(ll, CRqCreationLine) and ll.isPrefetch:
        if ll.isNeverAccessed and not ll.isNeverAccessedBecausePerms:
            useless.append(ll.boundsLength)
        else:
            useful.append(ll.boundsLength)

In [5]:
for u, t in ((useless, "useless"), (useful, "usseful")):
    categories = Counter(u)
    print(f"{t}:")
    for s, c in sorted(categories.items()):
        print(f"\t{s}: {c}")

useless:
	8: 17
	16: 94
	24: 133
	32: 267
	40: 351
	48: 394
	56: 291
	64: 416
	72: 85
	80: 3668
	88: 44
	96: 5913
	112: 295
	128: 119
	144: 105
	160: 62
	176: 56
	320: 3
	1840: 12
	2048: 898
	4096: 13
	131328: 1
	36893488147419103231: 22
usseful:
	4: 10
	8: 14
	16: 43
	24: 137
	32: 385
	40: 304
	48: 376
	56: 333
	64: 388
	72: 137
	80: 1146
	88: 73
	96: 3330
	112: 303
	128: 142
	144: 124
	160: 60
	176: 61
	320: 1
	1840: 224
	2048: 3578
	4096: 59
	4128: 38
	131328: 2
	36893488147419103231: 1


In [6]:
sum(lp.dists[CapPtrCacheDataArrivalLine]["chainedPrefetches"])/sum(lp.dists[CapPtrCacheDataArrivalLine]["nPrefetches"])

0.49666514768631675

In [7]:
nPrefetches = [x for x in lp.dists[CapPtrCacheDataArrivalLine]["nPrefetches"] if x != 0]
sum(nPrefetches)/len(nPrefetches)

1.5539461825672718

In [11]:
dueToPrefetch = 0
dueToAny = 0
for ll in lp.logLines:
    if isinstance(ll, CRqHitLine) and ll.cRqIsPrefetch and ll.cRqCreationLine.isNeverAccessed:
        if ll.evictionLine:
            if isinstance(ll.evictionLine, CRqMissLine) and ll.evictionLine.cRqIsPrefetch:
                dueToPrefetch += 1
            dueToAny += 1
dueToPrefetch/dueToAny

0.5230048106574565

In [15]:
lifetimes = []
for ll in lp.logLines:
    if isinstance(ll, CRqHitLine) and ll.timestamp > 2000000 and ll.cRqIsPrefetch and ll.cRqCreationLine.isNeverAccessed:
        if ll.evictionLine: 
            lifetimes.append(ll.evictionLine.timestamp - ll.timestamp)
sum(lifetimes)/len(lifetimes)


2935.71961047683

In [14]:
lifetimes = []
for ll in lp.logLines:
    if isinstance(ll, CRqHitLine) and ll.timestamp > 2000000:
        if ll.evictionLine: 
            lifetimes.append(ll.evictionLine.timestamp - ll.timestamp)
sum(lifetimes)/len(lifetimes)

3151.092911422465